# Whisper 사투리 파인튜닝 (MacBook Pro M4 Pro / MPS 로컬 실행용) — v2

지금까지 실행하면서 나온 문제들을 전부 반영한 통합 버전입니다.

반영된 수정:
1. 실제 AI-Hub 폴더 구조(Training/01.원천데이터, 02.라벨링데이터, 지역/시나리오 하위폴더)에 맞춘 재귀 탐색
2. 원천/라벨 파일명 접두어(`st_`/`lb_`)가 달라도 매칭되는 `norm_key` 방식
3. 손상된 라벨 json 파일은 건너뛰고 계속 진행 (전체가 멈추지 않도록)
4. `label_index`를 build_index 호출 전에 반드시 같은 셀에서 생성 (NameError 방지)
5. 서브셋을 쓸 경우 앞에서부터 자르지 않고 랜덤 샘플링 (강원도/경상도 편향 방지)
6. 라벨 토큰 길이 448 초과 시 truncation 처리 (Whisper 디코더 최대 길이 초과 에러 방지)
7. dataloader_num_workers 상향, 체크포인트 저장 빈도/보관개수 조정

**셀을 위에서부터 순서대로 실행하세요.**


In [ ]:
import os

# AI-Hub 데이터 루트 (Training/Validation 상위 폴더)
BASE_DIR = os.path.expanduser(
    "~/Desktop/1_Project/0_충북대학교 석사 1학기/4_학회/260406_ICCAS/CalmChat/"
    "notebooks/139-1.중·노년층 한국어 방언 데이터 (강원도, 경상도)"
)

audio_dir = os.path.join(BASE_DIR, "Training", "01.원천데이터")
label_dir = os.path.join(BASE_DIR, "Training", "02.라벨링데이터")

model_out_dir = os.path.expanduser("~/dialect_data/model/whisper-dialect")
os.makedirs(model_out_dir, exist_ok=True)

# 하위 폴더(강원도/경상도, 1인발화/질문답변 등)까지 전부 재귀 탐색
audio_files = []
for root, _, files in os.walk(audio_dir):
    for f in files:
        if f.endswith(".wav"):
            audio_files.append(os.path.join(root, f))

label_files = []
for root, _, files in os.walk(label_dir):
    for f in files:
        if f.endswith(".json"):
            label_files.append(os.path.join(root, f))

print(f"오디오 파일 수: {len(audio_files)}")
print(f"라벨 파일 수: {len(label_files)}")
print(audio_files[:3])


In [ ]:
!pip install -q -U transformers datasets librosa soundfile torch torchaudio evaluate
# 만약 이 셀 실행 후 WhisperProcessor import 에러(ModuleNotFoundError 등)가 나면
# torchvision 버전 충돌일 가능성이 큽니다. 아래 줄 주석 해제하고 실행 후 커널 재시작하세요.
# !pip uninstall -y torchvision


In [ ]:
import torch

# MPS(Apple Silicon GPU) 미지원 연산은 자동으로 CPU로 폴백
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"사용 디바이스: {device}")


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

print("모델 로드 완료")


## 데이터 인덱싱

- 오디오는 아직 로드하지 않고 (경로, 텍스트) 쌍만 모읍니다 (전체를 메모리에 올리면 OOM).
- 라벨 파일명이 오디오와 접두어(`st_`/`lb_`)만 다르고 나머지는 같다는 전제로, 접두어를 무시하고 매칭합니다.
- 라벨 json이 손상되어 파싱 실패하면 그 파일만 건너뛰고 계속 진행합니다.


In [ ]:
import json
import re

def norm_key(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    # 파일명 앞의 st_/lb_ 같은 접두어 차이를 무시하고 뒤쪽 고유 식별자로 비교
    return re.sub(r'^[a-zA-Z]+_', '', stem, count=1)

# label_index는 반드시 build_index 호출 전에 이 셀에서 만들어집니다.
label_index = {norm_key(p): p for p in label_files}

def build_index(audio_paths, label_index):
    items = []
    skipped = []

    for audio_path in sorted(audio_paths):
        label_path = label_index.get(norm_key(audio_path))
        if not label_path:
            continue

        try:
            with open(label_path, "r", encoding="utf-8") as f:
                label_data = json.load(f)
            text = label_data["transcription"]["standard"]
        except (json.JSONDecodeError, UnicodeDecodeError, KeyError) as e:
            skipped.append((label_path, str(e)))
            continue

        items.append({"audio_path": audio_path, "text": text})

    print(f"인덱싱된 데이터 수: {len(items)}")
    print(f"스킵된(손상/구조 다른) 라벨 수: {len(skipped)}")
    for path, err in skipped[:5]:
        print(f" - {path}\n   {err}")

    return items

dataset_index = build_index(audio_files, label_index)
print(f"샘플 텍스트: {dataset_index[0]['text']}")


In [ ]:
import random

random.seed(42)
SUBSET_SIZE = 60000  # 원하는 규모로 조정 (None이면 전체 사용)

if SUBSET_SIZE and SUBSET_SIZE < len(dataset_index):
    dataset_index = random.sample(dataset_index, SUBSET_SIZE)
    print(f"랜덤 서브셋 적용: {len(dataset_index)}개 (강원도/경상도 골고루 섞임)")
else:
    print(f"전체 데이터 사용: {len(dataset_index)}개")


In [ ]:
import librosa
from torch.utils.data import Dataset

class DialectDataset(Dataset):
    def __init__(self, items, processor):
        self.items = items
        self.processor = processor

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        # 오디오는 여기서(호출되는 시점에) 로드 -> 메모리에 전체를 안 쌓음
        speech, _ = librosa.load(item["audio_path"], sr=16000)
        inputs = self.processor(speech, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features[0]

        # Whisper 디코더는 최대 448 토큰까지만 처리 가능 -> 초과 시 잘라냄
        labels = self.processor.tokenizer(
            item["text"],
            return_tensors="pt",
            truncation=True,
            max_length=448,
        ).input_ids[0]

        return {"input_features": input_features, "labels": labels}

train_dataset = DialectDataset(dataset_index, processor)
print(f"데이터셋 크기: {len(train_dataset)}")


In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import evaluate

MAX_LABEL_LEN = 448

def collate_fn(batch):
    input_features = torch.stack([item["input_features"] for item in batch])

    label_list = [item["labels"] for item in batch]
    padded_labels = torch.full((len(label_list), MAX_LABEL_LEN), -100, dtype=torch.long)
    for i, label in enumerate(label_list):
        length = min(label.size(0), MAX_LABEL_LEN)
        padded_labels[i, :length] = label[:length]

    return {"input_features": input_features, "labels": padded_labels}

## 학습 설정

- `fp16=False` 유지: MPS는 CUDA용 fp16 학습(GradScaler)을 지원하지 않습니다. fp32로 진행합니다.
- `per_device_train_batch_size=8`부터 시도해보세요. `RuntimeError: MPS backend out of memory`가 나면 4 → 2로 낮추세요.
- `dataloader_num_workers`로 오디오 로딩을 병렬화합니다 (CPU 코어 활용, 병목이면 값 조정).
- `save_steps`/`save_total_limit`을 조정해 체크포인트가 디스크를 다 채우지 않도록 합니다.
- 전체 데이터/에폭 규모에 따라 학습 시간이 매우 오래 걸릴 수 있으니, `SUBSET_SIZE`와 `num_train_epochs`를 먼저 작게 잡고 한 번 끝까지 돌려보는 걸 권장합니다.


In [ ]:
import torch
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback

# 메모리 절약: activation을 저장 안 하고 backward 때 재계산
model.gradient_checkpointing_enable()
model.config.use_cache = False

class MPSCacheClearCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 50 == 0:
            torch.mps.empty_cache()

training_args = Seq2SeqTrainingArguments(
    output_dir=model_out_dir,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    dataloader_num_workers=0,       # 2 -> 0 : 별도 워커 프로세스 없이 메인 프로세스에서만 로딩
    dataloader_pin_memory=False,
    learning_rate=1e-5,
    save_steps=500,
    save_total_limit=3,
    logging_steps=20,
    predict_with_generate=True,
    fp16=False,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    processing_class=processor.feature_extractor,
    callbacks=[MPSCacheClearCallback()],
)

print("학습 시작!")
trainer.train()
print("학습 완료!")

: 

In [ ]:
model.save_pretrained(model_out_dir)
processor.save_pretrained(model_out_dir)
print(f"모델 저장 완료! -> {model_out_dir}")


In [ ]:
forced_decoder_ids = processor.get_decoder_prompt_ids(language="korean", task="transcribe")
model.config.forced_decoder_ids = forced_decoder_ids
model.to(device)

test_item = dataset_index[0]
speech, _ = librosa.load(test_item["audio_path"], sr=16000)
inputs = processor(speech, sampling_rate=16000, return_tensors="pt").to(device)

with torch.no_grad():
    predicted_ids = model.generate(inputs.input_features)
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)

print("정답:", test_item["text"])
print("예측:", transcription[0])
